# Ingeniería de Características (Feature Engineering)
## Proyecto de Tesis: Sistema inteligente para la detección y predicción de degradación de conectividad WiFi
**Autor:** Juan Vásquez  
**Universidad:** UDLA — Maestría en Inteligencia Artificial Aplicada  
**Fecha:** Marzo 2026

### Descripción
Este notebook transforma el dataset limpio mediante la creación de ventanas 
temporales y el etiquetado hacia adelante (forward labeling) para preparar 
los datos para el entrenamiento del modelo predictivo.

In [3]:
#IMPORTACION DE LIBRERIAS Y CARGA DEL DATASET LIMPIADO ANTERIORMENTE
import pandas as pd
import numpy as np

df = pd.read_csv(r"C:\Users\juanj\Desktop\Maestría\Tesis\Datasets\dataset_qos_limpio.csv")
print(df.shape)
print(df.head(5))

(7308, 11)
             timestamp  latencia_rtt_ms  jitter_ms  perdida_paquetes_pct  \
0  2026-03-16 00:00:01           18.111      0.766                   0.0   
1  2026-03-16 00:00:31           17.749      0.208                   0.0   
2  2026-03-16 00:01:01           17.853      0.500                   0.0   
3  2026-03-16 00:01:31           17.704      0.190                   0.0   
4  2026-03-16 00:02:01           18.088      0.662                   0.0   

   throughput_descarga_mbps  estado_latencia  estado_jitter  estado_perdida  \
0                    45.608                0              0               0   
1                    45.189                0              0               0   
2                    45.713                0              0               0   
3                    44.874                0              0               0   
4                    44.350                0              0               0   

   estado_throughput  num_umbrales_superados etiqueta_con

## 1. Parámetros de la ventana temporal

Se definen los parámetros de la ventana temporal para el horizonte base de 30 segundos (H1). El mismo proceso se repite para los horizontes H10 (5 min), H20 (10 min) y H30 (15 min) modificando únicamente el valor de HORIZONTE:

- **Ventana de entrada (VENTANA = 10)**: 10 muestras × 30 segundos = 5 minutos de historial previo. Cada muestra incorpora los valores de las 10 mediciones anteriores como contexto temporal para el modelo.
- **Horizonte de predicción (HORIZONTE = 1 / 10 / 20 / 30)**: define cuántas muestras hacia adelante se etiqueta el estado futuro. Para H1: 1 × 30 s = 30 segundos; para H10: 10 × 30 s = 5 minutos; para H20: 20 × 30 s = 10 minutos; para H30: 30 × 30 s = 15 minutos.

In [24]:
VENTANA = 10        # muestras hacia atrás (5 minutos de historial)
HORIZONTE = 30      # muestras hacia adelante (30 segundos de predicción)

metricas = ["latencia_rtt_ms", "jitter_ms", "perdida_paquetes_pct", "throughput_descarga_mbps"]

## 2. Creación de ventanas temporales

Se generan 40 nuevas columnas mediante desplazamiento temporal (shift) de los valores históricos de cada una de las cuatro métricas QoS con una ventana de 10 muestras hacia atrás. Esto produce columnas con nomenclatura `{metrica}_t-{i}` para i = 1 a 10, representando el valor de la métrica i muestras antes de la observación actual. El dataset pasa de 11 columnas originales a 52 columnas tras este proceso (11 + 40 nuevas = 51, más la de etiqueta futura que se agrega en el paso siguiente). Esta transformación permite al modelo aprender tendencias temporales en lugar de clasificar únicamente sobre valores instantáneos.

In [25]:
for i in range(1, VENTANA + 1):
    for metrica in metricas:
        df[f"{metrica}_t-{i}"] = df[metrica].shift(i)

print(df.shape)

(7308, 51)


## 3. Etiquetado hacia adelante (Forward Labeling)

Se asigna a cada fila la etiqueta de conectividad del estado que ocurrirá en el horizonte definido, mediante desplazamiento negativo del vector de etiquetas (`shift(-HORIZONTE)`). Este proceso convierte el problema de detección del estado actual en un problema de predicción proactiva: dadas las condiciones actuales y el historial reciente de la red, el modelo debe anticipar en qué estado se encontrará dentro de HORIZONTE × 30 segundos. Las primeras filas mantienen la etiqueta presente (normal) como etiqueta futura porque la red no ha cambiado de estado en ese instante. Las últimas HORIZONTE filas del dataset quedan sin etiqueta futura y serán eliminadas en el paso de limpieza.

In [26]:
df["etiqueta_futura"] = df["etiqueta_conectividad"].shift(-HORIZONTE)
df[["timestamp", "etiqueta_conectividad", "etiqueta_futura"]].head(40)

,timestamp,etiqueta_conectividad,etiqueta_futura
0,2026-03-16 00:00:01,normal,normal
1,2026-03-16 00:00:31,normal,normal
2,2026-03-16 00:01:01,normal,normal
3,2026-03-16 00:01:31,normal,normal
4,2026-03-16 00:02:01,normal,normal
5,2026-03-16 10:53:53,normal,normal
6,2026-03-16 10:54:23,normal,normal
7,2026-03-16 10:54:53,normal,normal
8,2026-03-16 10:55:23,normal,normal
9,2026-03-16 10:55:53,normal,normal


## 4. Eliminación de filas incompletas

Se eliminan mediante `dropna()` dos grupos de filas: las primeras 10 filas, que carecen de historial completo para la ventana de desplazamiento temporal, y las últimas HORIZONTE filas, que no tienen etiqueta futura asignada. Para el horizonte H1 (HORIZONTE=1) el dataset pasa de 5.289 a 5.249 filas, con la distribución de clases siguiente: normal 3.390 (64,7%), degradada 1.177 (22,4%) y crítica 682 (13,0%). La distribución se preserva correctamente tras la eliminación, lo que confirma que las filas descartadas no introducen sesgo de selección en el dataset final.

In [27]:
df_features = df.dropna().reset_index(drop=True)
print(df_features.shape)
print(df_features["etiqueta_futura"].value_counts())

(7268, 52)
etiqueta_futura
normal       5308
degradada     984
critica       976
Name: count, dtype: int64


## 5. Estadísticos de ventana deslizante

Se calculan cuatro estadísticos — media, desviación estándar, máximo y mínimo — de cada una de las cuatro métricas QoS sobre ventanas deslizantes de 10 y 30 muestras, generando 32 características adicionales con nomenclatura `{metrica}_{estadistico}_{ventana}`. Estos estadísticos capturan el comportamiento agregado de la serie en los últimos 5 minutos (ventana=10) y 15 minutos (ventana=30), complementando la información puntual del shift con información sobre la tendencia y variabilidad reciente. Tras este proceso el dataset alcanza las 84 columnas de características que constituyen el vector de entrada definitivo de los modelos.

In [28]:
ventanas = [10, 30]  # 10 muestras = 5 min, 30 muestras = 15 min

for v in ventanas:
    for metrica in metricas:
        df_features[f"{metrica}_mean_{v}"] = df_features[metrica].rolling(v).mean()
        df_features[f"{metrica}_std_{v}"]  = df_features[metrica].rolling(v).std()
        df_features[f"{metrica}_max_{v}"]  = df_features[metrica].rolling(v).max()
        df_features[f"{metrica}_min_{v}"]  = df_features[metrica].rolling(v).min()

print(df_features.shape)

(7268, 84)


## 6. Exportación del dataset final

Se exporta el dataset transformado con 84 características y etiqueta futura como archivo CSV. El proceso se repite cuatro veces cambiando únicamente el valor de HORIZONTE (1, 10, 20, 30) para generar los cuatro datasets de entrenamiento independientes correspondientes a cada horizonte de predicción del sistema. Los archivos exportados son: `Dataset_horizonte_1.csv` (H1, 30 s), `Dataset_horizonte_10.csv` (H10, 5 min), `Dataset_horizonte_20.csv` (H20, 10 min) y `Dataset_horizonte_30.csv` (H30, 15 min).

In [29]:
df_features.to_csv(r"C:\Users\juanj\Desktop\Maestría\Tesis\Datasets\Dataset_horizonte_30.csv", index = False)